# In your notebook cell
from adaptation.few_shot_adapt import main

if __name__ == "__main__":
    main()

In [2]:
# In your notebook cell
import sys
sys.path.append('/home/jovyan/DISSERTATION')

from adaptation.few_shot_adapt import main

main()


🔬 Starting Adaptation Experiment on cuda
   Config: 2-Shot | OBS_LEN=20
   Steps: Meta=50 vs Scratch=50 (Fair Budget)
   Latent Refine: Steps=25, LR=0.1
Loading checkpoint: checkpoints/meta_epoch_50.pt


RuntimeError: Error(s) in loading state_dict for TrajEncoder:
	size mismatch for rnn.weight_ih_l0: copying a param with shape torch.Size([384, 10]) from checkpoint, the shape in current model is torch.Size([192, 5]).
	size mismatch for rnn.weight_hh_l0: copying a param with shape torch.Size([384, 128]) from checkpoint, the shape in current model is torch.Size([192, 64]).
	size mismatch for rnn.bias_ih_l0: copying a param with shape torch.Size([384]) from checkpoint, the shape in current model is torch.Size([192]).
	size mismatch for rnn.bias_hh_l0: copying a param with shape torch.Size([384]) from checkpoint, the shape in current model is torch.Size([192]).
	size mismatch for rnn.weight_ih_l1: copying a param with shape torch.Size([384, 128]) from checkpoint, the shape in current model is torch.Size([192, 64]).
	size mismatch for rnn.weight_hh_l1: copying a param with shape torch.Size([384, 128]) from checkpoint, the shape in current model is torch.Size([192, 64]).
	size mismatch for r

In [3]:
# Run this in notebook to check:
from config.base_config import cfg
print(f"x_dim: {cfg.basis.x_dim}")
print(f"latent_dim: {cfg.latent.latent_dim}")
print(f"encoder_hidden: {cfg.latent.encoder_hidden_dim}")


x_dim: 5
latent_dim: 8
encoder_hidden: 64


In [4]:
# Kill all cached modules
import sys
to_remove = [key for key in sys.modules.keys() if 'config' in key or 'adaptation' in key or 'neural' in key or 'encoder' in key or 'head' in key]
for key in to_remove:
    del sys.modules[key]

print("✓ Cleared module cache")

# Fresh import
from config.base_config import cfg
print(f"✓ x_dim: {cfg.basis.x_dim}")
print(f"✓ latent_dim: {cfg.latent.latent_dim}")
print(f"✓ encoder_hidden: {cfg.latent.encoder_hidden_dim}")


✓ Cleared module cache
✓ x_dim: 10
✓ latent_dim: 16
✓ encoder_hidden: 128


In [6]:
# Clear ALL module caches more aggressively
import sys
import importlib

# Remove ALL cached modules
mods_to_clear = [m for m in list(sys.modules.keys()) if 'config' in m or 'dataloader' in m or 'trajectory' in m or 'adaptation' in m]
for m in mods_to_clear:
    del sys.modules[m]

print(f"Cleared {len(mods_to_clear)} modules")

# Reimport everything fresh
from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from adaptation.few_shot_adapt import main

print(f"✓ Config x_dim: {cfg.basis.x_dim}")
print(f"✓ Ready to run!")

main()


Cleared 7 modules
✓ Config x_dim: 10
✓ Ready to run!
🔬 Starting Adaptation Experiment on cuda
   Config: 2-Shot | OBS_LEN=20
   Steps: Meta=50 vs Scratch=50 (Fair Budget)
   Latent Refine: Steps=25, LR=0.1
Loading checkpoint: checkpoints/meta_epoch_50.pt

--- Processing testA ---


Adapting testA: 100%|██████████| 30/30 [06:19<00:00, 12.65s/it]



--- Processing testB ---


Adapting testB: 100%|██████████| 30/30 [06:07<00:00, 12.25s/it]



--- Processing testC ---


Adapting testC: 100%|██████████| 30/30 [06:03<00:00, 12.10s/it]



✅ Forecasting Adaptation Complete!

Average PATH MSE (Full Trajectory Physics):


OptionError: No such keys(s): 'display.width'

In [8]:
import os
import glob

# Find all CSV files
csvs = glob.glob("**/*.csv", recursive=True)
print("CSV files found:")
for csv in csvs:
    print(f"  {csv}")

# Check current directory
print(f"\nCurrent directory: {os.getcwd()}")
print("Files in current dir:")
for f in os.listdir(".")[:20]:
    print(f"  {f}")



CSV files found:
  data/index.csv
  results/adaptation_results.csv
  results/adaptation_with_distances.csv

Current directory: /home/jovyan/DISSERTATION 
Files in current dir:
  .ipynb_checkpoints
  sanity2.ipynb
  models
  sde_basis
  adaptation
  plots
  adapt_sanity.ipynb
  evaluation_sanity.ipynb
  training_sanity.ipynb
  dataloaders
  evaluation
  data
  results
  training
  data_gen
  training.log
  checkpoints
  sanity1.py
  config


In [10]:
import pandas as pd
import numpy as np

# Read the data
df = pd.read_csv("results/adaptation_results.csv")

# Bypass pandas display issues - use numpy
print("\n✅ Forecasting Adaptation Complete!\n")
print("="*80)
print("Average PATH MSE (Full Trajectory Physics):")
print("="*80)

for regime in ["testA", "testB", "testC"]:
    mask = df["regime"] == regime
    mse_zero = df[mask]["mse_path_zeroshot"].mean()
    mse_few = df[mask]["mse_path_fewshot"].mean()
    mse_scratch = df[mask]["mse_path_scratch"].mean()
    print(f"{regime:8s} | Zero-Shot: {mse_zero:.6f} | Few-Shot: {mse_few:.6f} | From-Scratch: {mse_scratch:.6f}")

print("\n" + "="*80)
print("Average FINAL-STEP MSE (Forecasting):")
print("="*80)

for regime in ["testA", "testB", "testC"]:
    mask = df["regime"] == regime
    mse_zero = df[mask]["mse_head_zeroshot"].mean()
    mse_few = df[mask]["mse_head_fewshot"].mean()
    mse_scratch = df[mask]["mse_head_scratch"].mean()
    print(f"{regime:8s} | Zero-Shot: {mse_zero:.6f} | Few-Shot: {mse_few:.6f} | From-Scratch: {mse_scratch:.6f}")

print("\n" + "="*80)
print("Summary Statistics:")
print("="*80)
print(f"Total tasks evaluated: {len(df)}")
print(f"Regimes: {df['regime'].unique()}")




✅ Forecasting Adaptation Complete!

Average PATH MSE (Full Trajectory Physics):
testA    | Zero-Shot: 0.208558 | Few-Shot: 0.208571 | From-Scratch: 0.403132
testB    | Zero-Shot: 0.417241 | Few-Shot: 0.417248 | From-Scratch: 0.626351
testC    | Zero-Shot: 0.695437 | Few-Shot: 0.695482 | From-Scratch: 0.928665

Average FINAL-STEP MSE (Forecasting):
testA    | Zero-Shot: 0.567442 | Few-Shot: 0.270567 | From-Scratch: 0.590878
testB    | Zero-Shot: 1.273147 | Few-Shot: 0.636951 | From-Scratch: 1.886285
testC    | Zero-Shot: 2.281282 | Few-Shot: 0.815658 | From-Scratch: 3.651459

Summary Statistics:
Total tasks evaluated: 90
Regimes: ['testA' 'testB' 'testC']


In [1]:
%load_ext autoreload
%autoreload 2

from adaptation import few_shot_adapt

few_shot_adapt.main()


🔬 Meta-Learning Adaptation (FIXED - ALL LAYERS FINE-TUNED)
Config: 2-Shot | OBS_LEN=20 | Steps=50
Strategy: Fine-tune Encoder+SDE+Head (FAIR COMPARISON with transfer)

Loading checkpoint: checkpoints/meta_epoch_50.pt
🎯 Processing testA
📊 Found 30 tasks



Adapting testA:   0%|          | 0/30 [00:00<?, ?it/s]/opt/conda/lib/python3.11/site-packages/torch/nn/modules/rnn.py:1133: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at ../aten/src/ATen/native/cudnn/RNN.cpp:1424.)
  result = _VF.gru(input, hx, self._flat_weights, self.bias, self.num_layers,
Adapting testA: 100%|██████████| 30/30 [09:44<00:00, 19.49s/it]


🎯 Processing testB
📊 Found 30 tasks



Adapting testB:   0%|          | 0/30 [00:00<?, ?it/s]/opt/conda/lib/python3.11/site-packages/torch/nn/modules/rnn.py:1133: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at ../aten/src/ATen/native/cudnn/RNN.cpp:1424.)
  result = _VF.gru(input, hx, self._flat_weights, self.bias, self.num_layers,
Adapting testB: 100%|██████████| 30/30 [10:09<00:00, 20.33s/it]


🎯 Processing testC
📊 Found 30 tasks



Adapting testC:   0%|          | 0/30 [00:00<?, ?it/s]/opt/conda/lib/python3.11/site-packages/torch/nn/modules/rnn.py:1133: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at ../aten/src/ATen/native/cudnn/RNN.cpp:1424.)
  result = _VF.gru(input, hx, self._flat_weights, self.bias, self.num_layers,
Adapting testC: 100%|██████████| 30/30 [09:50<00:00, 19.70s/it]


✅ Meta-Learning Adaptation Complete!

📊 RESULTS SUMMARY
--------------------------------------------------------------------------------

📈 Average PATH MSE (Full Trajectory Physics):
        mse_path_zeroshot  mse_path_fewshot
regime                                     
testA            0.158626          0.096638
testB            0.539714          0.189402
testC            1.000636          0.268577

📈 Average FINAL-STEP MSE (Forecasting):
        mse_head_zeroshot  mse_head_fewshot
regime                                     
testA            0.390163          0.286693
testB            1.674753          0.618912
testC            3.152204          0.846295

💾 Results saved to: results/adaptation_results.csv
